# 075c NX 同分异构打分（072d 仪器上的外部模型验证）

用 `ahmedberatozer/casmi26-nx-model` 给 072d 留出仪器的 150×25 候选打分。
不重训、不改候选集；只输出每个候选的 NX 分数，供本地做同式组 z 重排评估。


In [ ]:
import os, sys, glob, json, math, base64, zlib, io, time
from pathlib import Path
import subprocess
import numpy as np, pandas as pd

print('python', sys.version.split()[0])
whl = glob.glob('/kaggle/input/**/*rdkit*.whl', recursive=True)
print('rdkit wheels', whl[:3])
if whl:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    '--find-links', os.path.dirname(whl[0]), 'rdkit'], check=False)

CODE = Path('/kaggle/input/casmi26-v4b-models/code')
print('code dir', CODE.exists(), sorted(p.name for p in CODE.glob('*'))[:6])
sys.path.insert(0, str(CODE))
import torch, torch.nn as nn, torch.nn.functional as F
from casmi import fpnet, chem
from casmi.spectra import merge_spectra, clean_peaks
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
class NX(fpnet.FPNet):
    def __init__(self, nbits, d=512, layers=8, heads=8, emb=512):
        super().__init__(nbits, d=d, layers=layers, heads=heads)
        self.emb_dim = emb
        self.sproj = nn.Sequential(nn.Linear(2 * d, 1024), nn.GELU(), nn.Linear(1024, emb))
        self.tower = nn.Sequential(nn.Linear(nbits, 1024), nn.GELU(), nn.Dropout(0.1),
                                   nn.Linear(1024, 1024), nn.GELU(), nn.Linear(1024, emb))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(20.0)))

    def spec(self, *inp):
        z, el, e = self.forward(*inp, return_emb=True)
        return F.normalize(self.sproj(e).float(), dim=-1)

    def mol(self, b):
        return F.normalize(self.tower(b).float(), dim=-1)


device = 'cuda'
bits = np.load('/kaggle/input/casmi26-v2-pool/fp_bits.npy')
print('bits', bits.shape, bits.dtype, bits[:5])

CAND_B64 = "..."
sub = pd.read_csv(io.BytesIO(zlib.decompress(base64.b64decode(CAND_B64))))
sub['molecule_id'] = sub.molecule_id.astype(str)
ids = set(sub.molecule_id)
print('candidates', sub.shape, 'molecules', len(ids))

comp = None
for p in glob.glob('/kaggle/input/competitions/*'):
    if os.path.isfile(os.path.join(p, 'train.parquet')):
        comp = p
assert comp, 'competition data not found'
import pyarrow.parquet as pq
f = pq.ParquetFile(os.path.join(comp, 'train.parquet'))
t = f.read_row_group(0, columns=['inchikey14', 'adduct', 'precursor_mz', 'ms2_mzs',
                                 'ms2_normalized_intensities', 'ionization_mode', 'collision_energy_ev'])
q = t.to_pandas()
q = q[q.inchikey14.astype(str).isin(ids)].copy()
print('query rows', len(q), 'molecules', q.inchikey14.nunique())
assert q.inchikey14.nunique() == len(ids), (q.inchikey14.nunique(), len(ids))
qs = {m: g for m, g in q.groupby(q.inchikey14.astype(str), sort=False)}

fcache, bcache = {}, {}
def formula(s):
    if s not in fcache:
        p = chem.mol_props(s); fcache[s] = p[0] if p else None
    return fcache[s]

def cand_bits(s):
    if s not in bcache:
        fp = chem.raw_fingerprint(s)
        bcache[s] = None if fp is None else fp[bits].astype(np.float32)
    return bcache[s]

def queries_of(g):
    recs = []
    for r in g.itertuples(index=False):
        ce = r.collision_energy_ev
        ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)
        ce_arr = ce_arr[np.isfinite(ce_arr)]
        mz, it = clean_peaks(np.asarray(r.ms2_mzs, np.float64), np.asarray(r.ms2_normalized_intensities, np.float64),
                             float(r.precursor_mz), 0.001, 512, 2.0)
        if len(mz) == 0:
            continue
        recs.append(dict(mz=mz, it=it, prec=float(r.precursor_mz),
                         mode=1 if str(r.ionization_mode) == 'positive' else -1,
                         adduct_ix=int(chem.adduct_index(r.adduct)),
                         ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,
                         ce_n=int(len(ce_arr)) if len(ce_arr) else 1))
    items = []
    for mode in (1, -1):
        grp = [x for x in recs if x['mode'] == mode][:8]
        if not grp:
            continue
        if len(grp) == 1:
            mz, it = grp[0]['mz'], grp[0]['it']
        else:
            mz, it = merge_spectra([(x['mz'], x['it']) for x in grp])
        ces = np.array([x['ce'] for x in grp]); ces = ces[np.isfinite(ces)]
        pm, pi = fpnet.prep_peaks(mz, it, grp[0]['prec'])
        items.append(dict(mz=pm, it=pi, prec=grp[0]['prec'], adduct_ix=grp[0]['adduct_ix'],
                          ce=float(ces.mean()) if len(ces) else 0.0,
                          ce_known=1.0 if len(ces) else 0.0,
                          n_merged=min(int(sum(max(1, x['ce_n']) for x in grp)), 8),
                          mode=float(mode)))
    return items

rows = []
for tag, path in [('fold0', '/kaggle/input/casmi26-nx-model/nx_fold0_final.pt'),
                  ('full1', '/kaggle/input/casmi26-nx-model/nx_full1_final.pt')]:
    ck = torch.load(path, map_location='cpu', weights_only=False)
    model = NX(ck['nbits'], d=ck['d'], layers=ck['layers'], heads=ck['heads'], emb=ck['emb'])
    model.load_state_dict(ck['model']); model = model.to(device).eval()
    dt = fpnet.autocast_dtype(device)
    t0 = time.time(); n_act = 0
    with torch.no_grad():
        for mid, text in sub.itertuples(index=False):
            cands = text.split(';')
            if mid not in qs:
                continue
            items = queries_of(qs[mid])
            B = [cand_bits(s) for s in cands]
            if not items or any(b is None for b in B):
                continue
            inp = fpnet.collate(items, device)
            with torch.autocast(device, dtype=dt, enabled=device == 'cuda'):
                e = model.spec(*inp)
                m = model.mol(torch.as_tensor(np.stack(B), device=device))
            sc = (e @ m.T).float().mean(0).cpu().numpy()
            n_act += 1
            for i, s in enumerate(cands):
                rows.append(dict(molecule_id=mid, idx=i, smiles=s, score=float(sc[i]), model=tag))
    print(tag, 'scored molecules', n_act, 'seconds', round(time.time() - t0, 1))
    del model
    torch.cuda.empty_cache()

out = pd.DataFrame(rows)
out.to_csv('/kaggle/working/nx_scores.csv', index=False)
print('NX SCORES rows', len(out), 'molecules', out.molecule_id.nunique(),
      'models', out.model.unique().tolist())
print(out.groupby('model').score.describe().to_string())
